In [0]:
%sql
SELECT *
FROM sales_catalog.sales.orders;

In [0]:
%sql
SELECT
    city,
    SUM(amount) AS total_sales
FROM sales_catalog.sales.orders
GROUP BY city
ORDER BY total_sales DESC;

In [0]:
%sql SHOW SCHEMAS IN sales_catalog;


In [0]:
df = spark.table("sales_catalog.sales.orders")

display(df)


In [0]:
df.filter(df.amount > 50000).display()


In [0]:
df = spark.table("sales_catalog.sales.orders")

display(df)

In [0]:

spark.sql("DESCRIBE HISTORY sales_catalog.sales.orders").show()


In [0]:
from pyspark.sql.functions import sum

df = spark.table("sales_catalog.sales.orders")
df.groupBy("customer_name") \
  .agg(sum("amount").alias("total_spent")) \
  .show()

In [0]:
from pyspark.sql.functions import count

df = spark.table("sales_catalog.sales.orders")
df.groupBy("product") \
  .agg(count("order_id").alias("order_count")) \
  .show()

In [0]:
from pyspark.sql.functions import col

df = spark.table("sales_catalog.sales.orders")
df.filter(col("amount") > 50000) \
  .select("order_id", "customer_name", "product", "amount") \
  .show()

In [0]:
from pyspark.sql.functions import sum

df = spark.table("sales_catalog.sales.orders")
df.groupBy("city") \
  .agg(sum("amount").alias("total_sales")) \
  .orderBy("total_sales", ascending=False) \
  .show()

In [0]:
from pyspark.sql.functions import sum

df = spark.table("sales_catalog.sales.orders")
df.groupBy("city") \
  .agg(sum("amount").alias("total_sales")) \
  .orderBy("total_sales", ascending=True) \
  .show()

In [0]:
orders_df = spark.table("sales_catalog.sales.orders") # type: ignore
products_df = spark.table("sales_catalog.sales.products")

joined_df = orders_df.join(
    products_df,
    orders_df["product"] == products_df["product_name"],
    "inner"
).select(
    "order_id", 
    "customer_name", 
    "city", 
    "product", 
    "category", 
    "amount", 
    "order_date"
)

joined_df.show()

In [0]:
orders_df = spark.table("sales_catalog.sales.orders") # type: ignore
products_df = spark.table("sales_catalog.sales.products")

left_joined_df = orders_df.join(
    products_df,
    orders_df["product"] == products_df["product_name"],
    "left"
).select("order_id", "customer_name", "product", "category", "unit_price")
left_joined_df.show()

In [0]:
right_joined_df = orders_df.join(
    products_df,
    orders_df["product"] == products_df["product_name"],
    "right"
).select("order_id", "customer_name", "product_name", "category", "unit_price","product")

right_joined_df.show()

In [0]:
orders_df = spark.table("sales_catalog.sales.orders") # type: ignore
products_df = spark.table("sales_catalog.sales.products")

cross_joined_df = orders_df.crossJoin(products_df) \
    .select("order_id", "customer_name", "product", "product_name", "category")

cross_joined_df.show()

In [0]:
orders_df = spark.table("sales_catalog.sales.orders") # type: ignore

hyderabad_df = orders_df.filter(orders_df["city"] == "Hyderabad")
bangalore_df = orders_df.filter(orders_df["city"] == "Bangalore")

# unionByName ensures columns match by name rather than just position
union_all_df = hyderabad_df.unionByName(bangalore_df)
union_all_df.show()

In [0]:
hyderabad_df = orders_df.filter(orders_df["city"] == "Hyderabad").select("customer_name", "city","amount")

# union followed by distinct() removes duplicates
union_distinct_df = hyderabad_df.unionByName(hyderabad_df).distinct()
union_distinct_df.show()

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark session (already available as 'spark' in Databricks notebooks)
# spark = SparkSession.builder.getOrCreate()

# Read the tables from the catalog
customers_df = spark.table("sales_catalog.sales.customers")
orders_df = spark.table("sales_catalog.sales.orders")
products_df = spark.table("sales_catalog.sales.products")

# Perform the multi-table join
joined_df = (
    customers_df.alias("c")
    .join(
        orders_df.alias("o"),
        (customers_df["order_id"] == orders_df["order_id"]) & 
        (customers_df["customer_id"] == orders_df["order_id"]),
        "inner"
    )
    .join(
        products_df.alias("p"),
        customers_df["product_id"] == products_df["product_id"],
        "inner"
    )
    .select(
        "c.customer_id",
        "c.email",
        "c.phone_number",
        "o.order_id",
        "o.order_date",
        "o.city",
        "p.product_name",
        "p.category",
        "p.unit_price"
    )
)

# Display the results
display(joined_df)